In [1]:
#181 - folder structure

import os

def getDepth(path):
    return "    " * (len(path.split("\\")) - 2)

def removePath(path):
    return path.split("\\")[-1]

def showFiles(path):
    print(f"{getDepth(path)}{removePath(path)}")

    for item in os.listdir(path):
        fullPath = os.path.join(path, item)

        if os.path.isdir(fullPath):
            showFiles(fullPath)
        else:
            print(f"{getDepth(fullPath)}{removePath(fullPath)}")

showFiles(r"..\Project")

Project
    data
        landing
            customer_churn.csv
        logs
        raw
        rejected
    de1.ipynb
    ingestion.py


In [2]:
#182 - customer_churn.csv into data/landing/
import shutil

source = "../Project/customer_churn.csv"
destination = "../Project/data/landing/customer_churn.csv"

if os.path.exists(source):
    print("File exists")
    shutil.move(source, destination)

    print("File moved successfully")
else:
    print("File does not exist")

File does not exist


In [3]:
import os

def getDepth(path):
    return "    " * (len(path.split("\\")) - 2)

def removePath(path):
    return path.split("\\")[-1]

def showFiles(path):
    print(f"{getDepth(path)}{removePath(path)}")

    for item in os.listdir(path):
        fullPath = os.path.join(path, item)

        if os.path.isdir(fullPath):
            showFiles(fullPath)
        else:
            print(f"{getDepth(fullPath)}{removePath(fullPath)}")

showFiles(r"..\Project")

Project
    data
        landing
            customer_churn.csv
        logs
        raw
        rejected
    de1.ipynb
    ingestion.py


In [4]:
#183 - list csv files

def detect_files():
    files = []

    for file in os.listdir(r"../Project/data/landing"):
        if file.endswith(".csv"):
            files.append(file)

    return files

print(detect_files())

['customer_churn.csv']


In [5]:
#184 - EXPECTED_COLS

import pandas as pd

EXPECTED_COLS = ["customerID","gender","SeniorCitizen","Partner","Dependents","tenure","PhoneService", 
"MultipleLines","InternetService","OnlineSecurity","OnlineBackup","DeviceProtection", 
"TechSupport","StreamingTV","StreamingMovies","Contract","PaperlessBilling","PaymentMethod", 
"MonthlyCharges","TotalCharges","Churn"] 

df = pd.read_csv("../Project/data/landing/customer_churn.csv")

def validate_schema(cols):
    not_valid = []
    for i in cols:
        if i not in EXPECTED_COLS:
            not_valid.append(i)
    return not_valid

lis = validate_schema(df.columns.to_list())
if len(lis) == 0:
    print("Satisfies all the columns")
else:
    print(lis)

Satisfies all the columns


In [7]:
#185 - load to stage

from sqlalchemy.engine import URL
from sqlalchemy import create_engine
import pandas as pd

user = "root"
password = "root"
host = "localhost"
port = 3306
database = "proj"

url = URL.create(
    drivername="mysql+pymysql",
    username=user,
    password=password,
    host=host,
    port=port,
    database=database
)

engine = create_engine(url=url)

def load_to_staging(filepath, engine): 
    df = pd.read_csv(filepath) 
    df.to_sql("stg_customer_raw", engine, if_exists="replace", index=False) 
    return len(df) 

filepath = "../Project/data/landing/customer_churn.csv"
length = load_to_staging(filepath, engine)
print(f"loaded data frame length {length}")

loaded data frame length 7043


In [9]:
#186
from datetime import datetime
from sqlalchemy import text

def log_ingestion(
    filename,
    status,
    rows,
    reason,      # keep parameter if required by assignment
    engine
):
    
    query = text("""
        INSERT INTO ingestion_log
        (
            source_file,
            loaded_at,
            row_count,
            distinct_customer_ids,
            status
        )
        VALUES
        (
            :source_file,
            :loaded_at,
            :row_count,
            :distinct_customer_ids,
            :status
        )
    """)

    with engine.begin() as connection:
        connection.execute(
            query,
            {
                "source_file": filename,
                "loaded_at": datetime.now(),
                "row_count": rows,
                "distinct_customer_ids": rows,  # 7043 for this dataset
                "status": status
            }
        )

In [10]:
#187 - process_landing

def process_landing(engine):
    files = detect_files()

    for file in files:
        filepath = os.path.join("../Project/data/landing", file)

        valid = validate_schema(df.columns.to_list())

        if valid == []:
            rows = load_to_staging(filepath, engine)

            log_ingestion(file, "SUCCESS", rows, None, engine)

        else:
            log_ingestion(file, "FAILED", 0, "Validation failed", engine)

In [11]:
#188 - Tested

process_landing(engine)

### 189 - verify

mysql> select * from ingestion_log;

| id | table_name         | load_time           | row_count |
|----|--------------------|---------------------|-----------|
|  1 | stg_customer_raw   | 2026-08-13 19:39:15 |      7043 |
|  2 | customer_churn.csv | 2026-08-16 20:29:50 |      1000 |
|  3 | customer_churn.csv | 2026-08-16 20:32:47 |      7043 |
|  4 | customer_churn.csv | 2026-08-16 20:36:30 |         0 |
|  5 | customer_churn.csv | 2026-08-16 20:38:04 |      7043 |
|  6 | customer_churn.csv | 2026-08-16 20:38:43 |      7043 |

6 rows in set (0.00 sec)

